# Notebook 01 — SimpleMLP 784→8→4→2 on MNIST even/odd (digits 0, 1, 3, 4)

**Architecture:** SimpleMLP with hidden dims [8, 4], trained on MNIST digits 0, 1, 3, 4 with even/odd binary labels.

## Structure
0. **Imports & setup**
1. **Configuration**
2. **Backward Factor Trace** — seed-0 model, BFT, exploratory plots (pixel receptive fields, scaffold graphs)
3. **BFT figures** — main-paper figure 2 and its appendix companion
4. **Fingerprints** — NNLS factor tree, fingerprint similarity, near-OOD (excluded digits), far-OOD (synthetic)
5. **Fingerprint figures** — main paper and appendix (placeholders)

Validation, robustness and ablation analyses live in notebook 09.

## §0 — Imports & setup

In [ ]:
#%matplotlib inline
import sys, os
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets
from torchvision.transforms import ToTensor

from src import (
    SimpleMLP, load_experiment, get_transform, get_loaders_from_config,
    collect_layer_inputs_generic, collect_layer_dicts, bft, evaluate, build_scaffold_edges,
    scaffold_loading_from_edges, plot_scaffold_graph, extract_tree_nodes,
    extract_factor_tree_nodes, extract_fingerprint_matrix, compute_stimulus_similarity,
    project_stimuli_onto_tree, compute_factor_activations, plot_factor_tree,
    plot_input_layer_factors, plot_embedding_comparison, save_experiment,
)
from src.training import train_epoch, label_transform_even_odd
from src.data_utils import get_mnist_loaders, label_transformed_loader

matplotlib.rcParams.update({'figure.dpi': 80})
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

## §1 — Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
_nb_dir   = os.path.dirname(os.path.abspath('__file__'))
_repo_dir = os.path.dirname(_nb_dir)

MODEL_ROOT = os.path.join(_repo_dir, 'data', 'models')
FIG_DIR    = os.path.join(_repo_dir, 'figs', '01_mlp_8_4_0134')
for d in [MODEL_ROOT, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Experiment ────────────────────────────────────────────────────────────────
EXP_BASE     = 'mnist_even_odd_mlp_8_4_0134'
N_EPOCHS     = 30
DIGIT_FILTER = [0, 1, 3, 4]
N_CLASSES    = 2
CLASS_NAMES  = {0: 'even', 1: 'odd'}
IMAGE_SIDE   = 28

BASE_CONFIG = {
    'arch': 'SimpleMLP',
    'arch_kwargs': {'input_dim': 784, 'hidden_dims': [8, 4], 'output_dim': 2},
    'dataset': 'MNIST',
    'dataset_kwargs': {'root': '../data/', 'batch_size': 32,
                       'digit_filter': DIGIT_FILTER},
    'label_transform': 'even_odd',
    'analysis_layer_indices': [2, 4, 6],
    'n_per_class': 1000,
    'input_side': 28,
}

# ── BFT config — PUBLICATION SETTINGS (see archive/PUBLICATION_SETTINGS.md) ───────────
# From the nb09 S10 sweep (`rank x0.7`, run 4). The sweep passed [4, 1, 1], but bft's
# _auto_k_factorize floors k_max at 2 (`max(int(k_max), 2)`), so 1 and 2 behave identically
# here -- verified bit-identical fingerprints. [4, 2, 2] is written instead because it is
# what actually executes.
#
# Net effect vs the old [5, 5, 5]: ONLY the input layer changes (rank 5 -> 4). The root and
# hidden layers already auto-selected K = 2 / 1 / 2 and are untouched. Measured:
#   silhouette  0.666 -> 0.711     fingerprint  15 -> 13 dims
#   median causal R2  0.968 -> 0.955   BUT worst-node R2  0.778 -> 0.838 (improves)
N_BRANCHES         = [1, 2, 4]   # C0 circuit tree (nb15)
K_MAX_PER_LAYER    = [7, 6, 4]   # C0 circuit tree (nb15)
STIMULUS_THRESHOLD = 0.5
# Causal-validation sample count. The default (100) is what produced the spurious
# R2 = -0.484 input node; at 2000 the same node reconstructs at +0.778.
VALIDATE_TOP_M     = 2000

# ── Fingerprint / OOD config (§4) ─────────────────────────────────────────────
N_FAR_OOD = 300

print(f'MODEL_ROOT: {MODEL_ROOT}')
print(f'FIG_DIR:    {FIG_DIR}')

# Single-tree design: the fingerprint is the circuit tree's top-2 slice (§4);
# no separate fingerprint tree is fitted.


## §2 — Backward Factor Trace (seed 0)

### 2a — Model, loaders and layer activations

In [ ]:
# ── Seed-0 model: load or train ───────────────────────────────────────────────
exp_dir = os.path.join(MODEL_ROOT, f'{EXP_BASE}_seed0')

if os.path.exists(os.path.join(exp_dir, 'weights.pt')):
    model, config = load_experiment(exp_dir, DEVICE)
    print(f'Loaded model from {exp_dir}')
else:
    print('No checkpoint found, training from scratch...')
    torch.manual_seed(0)
    np.random.seed(0)
    model  = SimpleMLP(**BASE_CONFIG['arch_kwargs']).to(DEVICE)
    config = dict(BASE_CONFIG, description=f'{EXP_BASE} seed 0')

    _train_loader, _ = get_mnist_loaders(batch_size=32, root='../data/',
                                         digit_filter=DIGIT_FILTER)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    for ep in range(N_EPOCHS):
        train_epoch(model, _train_loader, optimizer, criterion, DEVICE,
                    label_transform_even_odd)
    save_experiment(model, config, exp_dir)
    print(f'Saved to {exp_dir}')

train_loader, test_loader = get_loaders_from_config(config)
label_transform = get_transform(config['label_transform'])
digit_filter    = config['dataset_kwargs']['digit_filter']
_, test_acc     = evaluate(model, test_loader, nn.CrossEntropyLoss(), DEVICE, label_transform)

linear_indices = model.linear_layer_indices()
LAYER_SIZES    = [model.layers[li].out_features for li in linear_indices]
print(f'Seed-0 test accuracy: {test_acc:.4f}')
print(f'Architecture (per-layer neurons): {LAYER_SIZES}')

In [ ]:
from src.data_utils import label_transformed_loader

# Loader yielding even/odd labels, for BFT primary mode (enables validate=True).
val_loader = label_transformed_loader(test_loader, label_transform)

# Collect the SAME samples BFT primary mode uses (loader order, only_correct), so
# downstream per-sample arrays stay aligned with tree_root's img_factors.
_collected  = collect_layer_dicts(model, test_loader, label_transform=label_transform,
                                  device=DEVICE)
all_images   = _collected['images']                              # (N, 1, 28, 28)
all_targets  = _collected['targets']                             # (N,) even/odd labels
all_digits   = _collected['digits']                              # (N,) original digit
layer_inputs = [d['input_fmap'] for d in _collected['layer_data']]  # list[(N, n_in)]
n_samples    = len(all_images)
flat_imgs    = all_images.reshape(n_samples, -1)

print(f'Samples: {n_samples}')
print(f'Class distribution: {dict(zip(*np.unique(all_targets, return_counts=True)))}')
for i, li in enumerate(layer_inputs):
    print(f'  L{i+1} inputs: {li.shape}')

### 2b — Run BFT

In [ ]:

# Primary mode: pass (model, loader) so BFT collects internally and validate=True works.
from src import cached_tree
tree_root = cached_tree('nb01_circuit', lambda: bft(
    model, val_loader, k_max=K_MAX_PER_LAYER, n_branches=N_BRANCHES,
    stimulus_threshold=STIMULUS_THRESHOLD, weighting='img_selectivity',
    validate=True, validate_top_m=VALIDATE_TOP_M, verbose=1, n_jobs=3),
    params=dict(k=K_MAX_PER_LAYER, b=N_BRANCHES, tau=STIMULUS_THRESHOLD, n=len(all_targets)))
print('Validation summary:', tree_root.validation_summary())


def get_nodes_by_depth(root):
    from collections import deque
    by_depth = {}
    queue = deque([(root, 0)])
    while queue:
        node, d = queue.popleft()
        by_depth.setdefault(d, []).append(node)
        for child in node.children:
            queue.append((child, d+1))
    return by_depth

def get_all_paths(root):
    if not root.children: return [[root]]
    return [[root] + p for child in root.children for p in get_all_paths(child)]

nodes_by_depth = get_nodes_by_depth(tree_root.root)
all_paths      = get_all_paths(tree_root.root)
print(f'Tree: {len(nodes_by_depth)} depths, {sum(len(v) for v in nodes_by_depth.values())} nodes, {len(all_paths)} paths')

### 2c — Exploratory plots: input-layer pixel receptive fields

In [ ]:
# ── Plot 2: Input-layer pixel receptive fields ────────────────────────────────
# Shows connection_factors[:, k] for each L1 node reshaped to (n_out, 28, 28)
max_depth = max(nodes_by_depth.keys())
for r in nodes_by_depth[max_depth]:
    path_label = 'F' + '→F'.join(str(f) for f in r.path) if r.path else 'root'
    figs = plot_input_layer_factors(
        r, all_images, arch='fc', image_shape=(IMAGE_SIDE, IMAGE_SIDE)
    )
    for k, fig in enumerate(figs):
        fname = f"pixel_rf_L{r.layer_idx+1}_{path_label.replace('→','-')}_k{k}.pdf"
        fig.savefig(os.path.join(FIG_DIR, fname), bbox_inches='tight')
        plt.show(); plt.close(fig)


### 2d — Exploratory plots: full-network scaffold graphs

In [ ]:
for path_nodes in all_paths:
    layer_results = list(reversed(path_nodes))  # L1-first order

    edge_matrices, neg_edge_matrices = build_scaffold_edges(
        layer_results[1:], fi='path', fi_seed=layer_results[0], top_pct=0.05,
    )
    loading = scaffold_loading_from_edges(edge_matrices)

    path_label = 'F' + '→F'.join(str(f) for f in path_nodes[-1].path)

    fig = plot_scaffold_graph(loading, edge_matrices, LAYER_SIZES,
                              neg_edge_matrices=neg_edge_matrices)
    fig.suptitle(f'Scaffold Graph [path: {path_label}]', fontsize=11)

    _pl = path_label.replace('→', '-')
    fig.savefig(os.path.join(FIG_DIR, f'scaffold_{_pl}.pdf'), bbox_inches='tight')
    plt.show()


## §3 — BFT figures (main paper & appendix)

### 3a — Main-paper figure 2 — class and sub-class circuits

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT — figure data for the 8x4 even/odd MLP circuits
# Requires: tree_root, all_digits, all_images, LAYER_SIZES  (§2 above)
# Writes:   figures/figdata/nb01_circuits.npz  (+ .json)
# The figures themselves are built in notebooks/fig01_mlp_even_odd.ipynb, which
# needs nothing but this bundle — no model, no MNIST, no BFT.
# ═══════════════════════════════════════════════════════════════════════════════
from src import figdata, figexport

DIGIT_ORDER = [0, 4, 1, 3]          # even digits first, then odd
IMAGE_SIDE  = 28


def digit_loading(node, k, digits=None, digit_order=DIGIT_ORDER):
    """Share of factor k's mean stimulus loading contributed by each digit."""
    digits = all_digits if digits is None else digits
    col = node.img_factors[:, k]
    m = np.array([col[digits == d].mean() for d in digit_order])
    return m / (m.sum() + 1e-12)


def digit_profiles(node):
    return np.stack([digit_loading(node, k) for k in range(node.img_factors.shape[1])])


def pixel_arbors(node, neg=False):
    """(K, 28, 28): pixel-space arbor of every factor of an input-layer node."""
    F = node.neg_connection_factors if neg else node.connection_factors
    n_out, n_in = node.weight.shape
    return np.stack([F[:, k].reshape(n_out, n_in).sum(0).reshape(IMAGE_SIDE, IMAGE_SIDE)
                     for k in range(F.shape[1])])


def conn_map(node, k, neg=False):
    """One factor's connection map, shaped (out unit, in unit)."""
    F = node.neg_connection_factors if neg else node.connection_factors
    return F[:, k].reshape(node.weight.shape)


def weighted_avg_stimuli(node, flat_images):
    """(K, 28, 28): the stimulus average each factor's loadings define."""
    W = node.img_factors
    return np.stack([(W[:, k, None] * flat_images).sum(0) / (W[:, k].sum() + 1e-12)
                     for k in range(W.shape[1])]).reshape(-1, IMAGE_SIDE, IMAGE_SIDE)


# ── index the trace tree; identify the even / odd root factor from the data ──
root = tree_root.root
by_path = {}


def _walk(n):
    by_path[tuple(n.path)] = n
    for ch in n.children:
        _walk(ch)


_walk(root)

_even_cols = [i for i, d in enumerate(DIGIT_ORDER) if d % 2 == 0]
root_even_k = int(np.argmax([digit_loading(root, k)[_even_cols].sum()
                             for k in range(root.img_factors.shape[1])]))
_flat = all_images.reshape(len(all_images), -1)

# ── the two class circuits, with everything either figure draws ─────────────
circuits = []
for name, k in (('even', root_even_k), ('odd', 1 - root_even_k)):
    l2, l1 = by_path[(k,)], by_path[(k, 0)]
    chain = list(reversed([root, l2, l1]))                      # L1-first
    E, negE = build_scaffold_edges(chain[1:], fi='path', fi_seed=chain[0], top_pct=0.05)
    cf0 = l1.connection_factors[:, 0].reshape(l1.weight.shape)
    circuits.append(dict(
        name=name, color_key=name, k=k,
        l3_profile=digit_loading(root, k),
        l3_conn=conn_map(root, k), l3_conn_neg=conn_map(root, k, neg=True),
        l2_lam=l2.lambdas / l2.lambdas.sum(), l2_profiles=digit_profiles(l2),
        l2_conn=conn_map(l2, 0), l2_conn_neg=conn_map(l2, 0, neg=True),
        l1_lam=l1.lambdas / l1.lambdas.sum(), l1_profiles=digit_profiles(l1),
        l1_arbors=pixel_arbors(l1), l1_neg_arbors=pixel_arbors(l1, neg=True),
        # per-unit receptive fields of *every* L1 factor, not just the dominant one
        l1_unit_rf_all=np.stack([l1.connection_factors[:, kk].reshape(l1.weight.shape)
                                 for kk in range(l1.img_factors.shape[1])]).reshape(
                                     -1, l1.weight.shape[0], IMAGE_SIDE, IMAGE_SIDE),
        l1_unit_rf=cf0.reshape(-1, IMAGE_SIDE, IMAGE_SIDE),
        l1_unit_share=cf0.sum(1) / (cf0.sum() + 1e-12),
        l1_wavg=weighted_avg_stimuli(l1, _flat),
        scaffold=figexport.scaffold_summary(E, negE, scaffold_loading_from_edges(E),
                                            LAYER_SIZES)))

# superset: the whole trace, the shared stimulus pool and a few real images,
# so panels can be redesigned later without re-running BFT
nodes = figexport.export_tree(root, labels=all_digits, classes=DIGIT_ORDER,
                              images=all_images)
D = figdata.save('nb01_circuits', dict(
    digit_order=DIGIT_ORDER, layer_sizes=LAYER_SIZES, out_labels=['even', 'odd'],
    root_lam=root.lambdas / root.lambdas.sum(),
    root_factor_color_keys=['even' if i == root_even_k else 'odd'
                            for i in range(root.img_factors.shape[1])],
    circuits=circuits,
    stim_labels=all_targets.astype(int), stim_digits=all_digits.astype(int),
    images=figexport.stimulus_pool(all_images, nodes, max_side=28),
    meta=figexport.trace_meta(tree_root, k_max=K_MAX_PER_LAYER,
                              n_branches=N_BRANCHES, stimulus_threshold=STIMULUS_THRESHOLD,
                              classes=DIGIT_ORDER, test_acc=float(test_acc)),
    nodes=nodes,
    stimuli=figexport.example_stimuli(all_images, all_digits, DIGIT_ORDER,
                                      per_class=8, max_side=28)))
figdata.summary('nb01_circuits')


### 3b — Appendix figure — decomposition details

In [ ]:
# (appendix figure moved to notebooks/fig01_mlp_even_odd.ipynb)


## §4 — Fingerprints

In [ ]:
# ── Single tree: the fingerprint is the circuit tree's top two levels — the
#    output-layer factors plus their immediate sub-circuits (see PUBLICATION.md).
from src import truncate_tree
tree_circuit = tree_root                      # the full circuit tree (§2)
tree_root = truncate_tree(tree_circuit, depth=2)
print('circuit tree:', sum(1 for _ in tree_circuit.nodes()), 'nodes | '
      'fingerprint (top-2 slice):', sum(1 for _ in tree_root.nodes()), 'nodes,',
      sum(n.img_factors.shape[1] for n in tree_root.nodes()), 'dims')

# Uniform analysis contract consumed by §6-§9 (identical cells across notebooks).
ANALYSIS_CTX = dict(
    exp='mlp_even_odd', prune_name='nb13_pruning_mlp_even_odd',
    tag='nb01', model=model, tree_circuit=tree_circuit, tree_fp=tree_root,
    layer_inputs=layer_inputs, labels_task=all_targets.astype(int),
    labels_fine=all_digits.astype(int), eval_loader=test_loader,
    label_transform=label_transform, device=DEVICE,
    layer_names=[d['name'] for d in _collected['layer_data']],
    n_classes=N_CLASSES, k_cap=8, last_extra=2, k_max_cfg=K_MAX_PER_LAYER,
    prune_fractions=(0.02, 0.05, 0.1, 0.2), n_random=5, stab_seeds=5)

In [ ]:
# §3 only exports plot data now — the paper figures live in
# notebooks/fig01_mlp_even_odd.ipynb — so nothing above changed matplotlib's
# rcParams. Kept so the exploratory plots below render at screen size.
plt.rcParams.update({'figure.dpi': 80})


### 4a — NNLS round-trip on the publication factor tree

In [ ]:
# ── Fingerprint basis = the publication tree from §2 ──────────────────────────
# §4 and §5 project new stimuli onto the SAME factors the paper's circuit figure
# shows, so a fingerprint dimension means the same thing as a node in Fig. 2.
# This section used to build a second, wider tree (k_max=[10,1,2],
# n_branches=[8,2,2], stimulus_threshold=0). Measured against it, the publication
# tree is better on every axis: round-trip cosine 0.994 ± 0.040 vs 0.740 ± 0.106,
# even/odd silhouette 0.890 vs 0.617, and 13 dimensions instead of 35.
fp_tree_nodes   = extract_tree_nodes(tree_root)
fp_factor_nodes = extract_factor_tree_nodes(tree_root)

# Round-trip test: NNLS-project the test set back onto the fixed factors and
# compare with the fingerprints the NMF itself produced.
from sklearn.metrics.pairwise import paired_cosine_distances as _pcd
projected_test_rt = project_stimuli_onto_tree(tree_root, layer_inputs)
F_orig  = extract_fingerprint_matrix(tree_root, np.arange(n_samples))
F_rt    = extract_fingerprint_matrix(projected_test_rt, np.arange(n_samples))
rt_sims = 1.0 - _pcd(F_orig, F_rt)
print(f'Round-trip cosine sim: mean={rt_sims.mean():.4f}  std={rt_sims.std():.4f}  min={rt_sims.min():.4f}')


### 4b — Factor fingerprints & similarity (test set)

In [ ]:
F = extract_fingerprint_matrix(tree_root, np.arange(n_samples))
print(f'Fingerprint matrix: {F.shape}')

MAX_VIZ = 300
viz_idx     = np.argsort(all_targets)[:MAX_VIZ] if n_samples > MAX_VIZ else np.argsort(all_targets)
S           = compute_stimulus_similarity(F[viz_idx])
targets_viz = all_targets[viz_idx]

# ── Similarity heatmap + intra/inter histogram ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im = axes[0].imshow(S, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
fig.colorbar(im, ax=axes[0])
for cl in np.unique(targets_viz):
    for b in np.where(np.diff((targets_viz == cl).astype(int)))[0]:
        axes[0].axhline(b + 0.5, color='k', lw=0.5)
        axes[0].axvline(b + 0.5, color='k', lw=0.5)
axes[0].set(title='Factor fingerprint similarity (sorted by class)',
            xlabel='Stimulus', ylabel='Stimulus')

S_all = compute_stimulus_similarity(F)
classes = sorted(np.unique(all_targets))
intra_vals, inter_vals = [], []
for ci, cl in enumerate(classes):
    mask  = all_targets == cl
    intra = S_all[np.ix_(mask, mask)]
    intra_vals.extend(intra[np.triu_indices_from(intra, k=1)])
    for cl2 in classes[ci + 1:]:
        inter_vals.extend(S_all[np.ix_(mask, all_targets == cl2)].ravel())
intra_arr, inter_arr = np.array(intra_vals), np.array(inter_vals)
axes[1].hist(intra_arr, bins=60, alpha=0.6, density=True,
             label=f'Intra ({intra_arr.mean():.3f})')
axes[1].hist(inter_arr, bins=60, alpha=0.6, density=True,
             label=f'Inter ({inter_arr.mean():.3f})')
axes[1].set(xlabel='Cosine similarity', ylabel='Density',
            title='Intra- vs inter-class fingerprint similarity')
axes[1].legend()

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'nnls_fingerprints.pdf'), bbox_inches='tight')
plt.show()

# ── Plot 7: Embedding comparison ───────────────────────────────────────────────
# Panels: PCA(fingerprints) | PCA(last-layer) | PCA(all-layers) | MDS(fingerprints)
full_acts_id = np.concatenate([li[viz_idx] for li in layer_inputs], axis=1)
fig = plot_embedding_comparison(
    F[viz_idx], layer_inputs[-1][viz_idx], targets_viz,
    CLASS_NAMES, digit_targets=all_digits[viz_idx],
    activations_all=full_acts_id,
    title='ID test-set fingerprint embeddings',
)
fig.savefig(os.path.join(FIG_DIR, 'embedding_comparison_id.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig)

### 4c — Near-OOD: excluded MNIST digits (2, 5, 6, 7, 8, 9)

In [ ]:
OOD_DIGITS = [d for d in range(10) if d not in digit_filter]
print(f'OOD digits: {OOD_DIGITS}')

mnist_test_full = datasets.MNIST('../data/', train=False, download=True, transform=ToTensor())
ood_labels_all  = np.array(mnist_test_full.targets)
ood_indices     = np.where(np.isin(ood_labels_all, OOD_DIGITS))[0]
ood_subset      = Subset(mnist_test_full, ood_indices)

data_ood = collect_layer_inputs_generic(
    model, ood_subset, label_transform=label_transform_even_odd,
    only_correct=False, device=DEVICE,
)
ood_images  = data_ood['images']
ood_digits  = data_ood['digits']
ood_targets = data_ood['targets']
ood_inputs  = data_ood['layer_inputs']
n_ood       = len(ood_images)
print(f'OOD samples: {n_ood}')

ood_preds        = data_ood['preds']   # what the network says, for §5
projected_ood    = project_stimuli_onto_tree(tree_root, ood_inputs)
factor_nodes_ood = extract_factor_tree_nodes(projected_ood)

# ── Factor tree per OOD digit ─────────────────────────────────────────────────
unique_ood = sorted(np.unique(ood_digits))
ncols = min(3, len(unique_ood))
nrows = int(np.ceil(len(unique_ood) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)
for i, d in enumerate(unique_ood):
    ax   = axes[i // ncols][i % ncols]
    idx  = np.where(ood_digits == d)[0]
    acts = compute_factor_activations(factor_nodes_ood, idx)
    parity = 'even' if d % 2 == 0 else 'odd'
    plot_factor_tree(factor_nodes_ood, acts, ax=ax,
                     title=f'OOD digit {d} ({parity})  n={len(idx)}')
for j in range(len(unique_ood), nrows * ncols):
    axes[j // ncols][j % ncols].set_visible(False)
plt.suptitle('Near-OOD — Factor tree per excluded digit', fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'near_ood_tree_per_digit.pdf'), bbox_inches='tight')
plt.show()

# ── Plot 7: Embedding comparison — ID test vs near-OOD ────────────────────────
N_EACH = 150
rng_m  = np.random.default_rng(7)
id_sub  = rng_m.choice(n_samples, min(N_EACH, n_samples), replace=False)
ood_sub = rng_m.choice(n_ood, min(N_EACH, n_ood), replace=False)

F_id_sub  = extract_fingerprint_matrix(tree_root, id_sub)
F_ood_sub = extract_fingerprint_matrix(projected_ood, ood_sub)
F_combo   = np.concatenate([F_id_sub, F_ood_sub])
act_combo = np.concatenate([layer_inputs[-1][id_sub], ood_inputs[-1][ood_sub]])
lbl_combo = np.concatenate([all_targets[id_sub], ood_targets[ood_sub]])
digit_combo = np.concatenate([all_digits[id_sub], ood_digits[ood_sub]])
cond_combo  = (['ID'] * len(id_sub)) + (['near-OOD'] * len(ood_sub))

full_id_sub  = np.concatenate([li[id_sub]  for li in layer_inputs], axis=1)
full_ood_sub = np.concatenate([li[ood_sub] for li in ood_inputs],   axis=1)
full_combo   = np.concatenate([full_id_sub, full_ood_sub],           axis=0)

fig = plot_embedding_comparison(
    F_combo, act_combo, lbl_combo,
    CLASS_NAMES, digit_targets=digit_combo,
    condition_labels=cond_combo,
    activations_all=full_combo,
    title='Near-OOD vs ID fingerprint embeddings',
)
fig.savefig(os.path.join(FIG_DIR, 'embedding_comparison_near_ood.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig)

### 4d — Far-OOD: synthetic images

In [ ]:
rng_f = np.random.default_rng(99)
_chk  = (np.indices((IMAGE_SIDE, IMAGE_SIDE)).sum(0) % 2).astype(np.float32)

far_ood_arrays = {
    'gaussian_noise': np.clip(
        rng_f.normal(0.5, 0.25, (N_FAR_OOD, 1, IMAGE_SIDE, IMAGE_SIDE)).astype(np.float32), 0, 1),
    'uniform_gray':   np.full((N_FAR_OOD, 1, IMAGE_SIDE, IMAGE_SIDE), 0.5, dtype=np.float32),
    'checkerboard':   np.broadcast_to(_chk, (N_FAR_OOD, 1, IMAGE_SIDE, IMAGE_SIDE)).copy().astype(np.float32),
    'inverted_test':  np.clip(1.0 - all_images[:N_FAR_OOD], 0, 1).astype(np.float32),
}

far_ood_data = {}
for name, imgs in far_ood_arrays.items():
    ds = TensorDataset(torch.from_numpy(imgs),
                       torch.zeros(len(imgs), dtype=torch.long))
    d = collect_layer_inputs_generic(
        model, ds, label_transform=label_transform_even_odd,
        only_correct=False, device=DEVICE,
    )
    d['projected_root'] = project_stimuli_onto_tree(tree_root, d['layer_inputs'])
    d['factor_nodes']   = extract_factor_tree_nodes(d['projected_root'])
    far_ood_data[name]  = d
    print(f'{name:20s}  model acc={(d["preds"] == d["targets"]).mean():.3f}')

# ── Factor tree per far-OOD type ──────────────────────────────────────────────
n_types = len(far_ood_data)
fig, axes = plt.subplots(1, n_types, figsize=(6 * n_types, 4.5), squeeze=False)
axes = axes[0]
for ax, (name, d) in zip(axes, far_ood_data.items()):
    acts = compute_factor_activations(d['factor_nodes'], np.arange(len(d['images'])))
    plot_factor_tree(d['factor_nodes'], acts, ax=ax, title=name)
plt.suptitle('Far OOD — factor tree activation per synthetic type', y=1.02, fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'far_ood_factor_tree.pdf'), bbox_inches='tight')
plt.show()

# ── Plot 7: Embedding comparison — ID / near-OOD / far-OOD ───────────────────
N_MDS = 80
rng_m2 = np.random.default_rng(5)

F_parts, act_parts, full_parts, lbl_parts, digit_parts, cond_parts = [], [], [], [], [], []

# ID test
id_s2 = rng_m2.choice(n_samples, min(N_MDS, n_samples), replace=False)
F_parts.append(extract_fingerprint_matrix(tree_root, id_s2))
act_parts.append(layer_inputs[-1][id_s2])
full_parts.append(np.concatenate([li[id_s2] for li in layer_inputs], axis=1))
lbl_parts.append(all_targets[id_s2])
digit_parts.append(all_digits[id_s2])
cond_parts.extend(['ID-test'] * len(id_s2))

# Near-OOD
ood_s2 = rng_m2.choice(n_ood, min(N_MDS, n_ood), replace=False)
F_parts.append(extract_fingerprint_matrix(projected_ood, ood_s2))
act_parts.append(ood_inputs[-1][ood_s2])
full_parts.append(np.concatenate([li[ood_s2] for li in ood_inputs], axis=1))
lbl_parts.append(ood_targets[ood_s2])
digit_parts.append(ood_digits[ood_s2])
cond_parts.extend(['near-OOD'] * len(ood_s2))

# Far-OOD types
for name, d in far_ood_data.items():
    n  = min(N_MDS, len(d['images']))
    ss = rng_m2.choice(len(d['images']), n, replace=False)
    F_parts.append(extract_fingerprint_matrix(d['projected_root'], ss))
    act_parts.append(d['layer_inputs'][-1][ss])
    full_parts.append(np.concatenate([li[ss] for li in d['layer_inputs']], axis=1))
    lbl_parts.append(d['targets'][ss])
    digit_parts.append(np.full(n, -1))
    cond_parts.extend([name] * n)

F_joint     = np.concatenate(F_parts)
act_joint   = np.concatenate(act_parts)
lbl_joint   = np.concatenate(lbl_parts)
digit_joint = np.concatenate(digit_parts)

fig = plot_embedding_comparison(
    F_joint, act_joint, lbl_joint,
    CLASS_NAMES, digit_targets=None,
    condition_labels=cond_parts,
    far_ood_conditions=list(far_ood_data.keys()),
    activations_all=np.concatenate(full_parts, axis=0),
    title='ID / near-OOD / far-OOD fingerprint embeddings',
)
fig.savefig(os.path.join(FIG_DIR, 'embedding_comparison_all_ood.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig)

## §5 — Fingerprint figures (main paper & appendix)

### 5a — Main-paper figure

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT — figure data for the 8x4 MLP fingerprints
# Requires: tree_root, F, all_targets, all_digits, layer_inputs  (§2, §4b)
#           projected_ood, ood_digits, ood_preds, n_ood  (§4c)
#           far_ood_data  (§4d) and rt_sims (§4a)
# Writes:   figures/figdata/nb01_fingerprints.npz  (+ .json)
# Figures are built in notebooks/fig01_mlp_even_odd.ipynb from this bundle alone.
# ═══════════════════════════════════════════════════════════════════════════════
from src import figdata, figexport
from src.paper_figures import unit
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

DIGIT_ORDER = [0, 4, 1, 3]                       # even digits first, then odd
FAR_LABEL   = {'gaussian_noise': 'noise', 'uniform_gray': 'gray',
               'checkerboard': 'checker', 'inverted_test': 'inverted'}

# ── fingerprints of the OOD conditions, on the §2 factors ───────────────────
F_ood = extract_fingerprint_matrix(projected_ood, np.arange(n_ood))
F_far = {name: extract_fingerprint_matrix(d['projected_root'],
                                          np.arange(len(d['images'])))
         for name, d in far_ood_data.items()}

# ── map every fingerprint dim to its (layer, root branch, factor) ───────────
# extract_fingerprint_matrix concatenates nodes in BFS order, so this reproduces
# the column order of F.
_dims, _q = [], [tree_root.root]
while _q:
    _n = _q.pop(0)
    for _k in range(_n.img_factors.shape[1]):
        _dims.append((_n.layer_idx, _n.path[0] if _n.path else -1, _k))
    _q.extend(_n.children)
_dims  = np.array(_dims)
_top   = _dims[:, 0].max()
ROOT_K = np.where(_dims[:, 0] == _top)[0]

# which root factor is the even one (read off the data, as in Fig. 2)
_is_even = all_targets == 0
K_EVEN = int(ROOT_K[np.argmax([F[_is_even, k].mean() - F[~_is_even, k].mean()
                               for k in ROOT_K])])
K_ODD  = int([k for k in ROOT_K if k != K_EVEN][0])


def _circuit_cols(k_root):
    """The fingerprint dims belonging to one root factor's circuit."""
    return [int(k_root)] + [i for i in range(len(_dims))
                            if _dims[i, 0] != _top and _dims[i, 1] == _dims[k_root, 2]]


COLS_EVEN, COLS_ODD = _circuit_cols(K_EVEN), _circuit_cols(K_ODD)
COL_ORDER = COLS_EVEN + COLS_ODD
N_EV = len(COLS_EVEN)


# ── what the fingerprint says vs what the network says ─────────────────────
def odd_share(Fm):
    """Share of the two root factors' mass carried by the odd circuit."""
    o, e = Fm[:, K_ODD], Fm[:, K_EVEN]
    return o / (o + e + 1e-12)


ood_pts = [(d, (ood_preds[ood_digits == d] == 1).mean(),
            odd_share(F_ood[ood_digits == d]).mean())
           for d in sorted(np.unique(ood_digits))]
id_pts  = [(d, (all_targets[all_digits == d] == 1).mean(),
            odd_share(F[all_digits == d]).mean()) for d in DIGIT_ORDER]
R_OOD = np.corrcoef([p[1] for p in ood_pts], [p[2] for p in ood_pts])[0, 1]


# ── how tightly each condition's fingerprints cluster ──────────────────────
def centroid_cos(Fm):
    """Cosine of every fingerprint to its own condition's mean fingerprint."""
    U = unit(Fm)
    c = U.mean(0)
    return U @ (c / (np.linalg.norm(c) + 1e-12))


COND = ([dict(label='ID test', values=centroid_cos(F), color_key='id_data'),
         dict(label='held-out digits', values=centroid_cos(F_ood),
              color_key='near_ood')] +
        [dict(label=FAR_LABEL[n], values=centroid_cos(F_far[n]), color_key='far_ood')
         for n in far_ood_data])

SIL_CLASS = silhouette_score(unit(F), all_targets, metric='cosine')
SIL_DIGIT = silhouette_score(unit(F), all_digits, metric='cosine')
print(f'fingerprint dim: {F.shape[1]}   silhouette: {SIL_CLASS:.3f} (even/odd), '
      f'{SIL_DIGIT:.3f} (digit)')
print(f'held-out digits: r(odd-circuit share, P(model=odd)) = {R_OOD:.4f}')
for c in COND:
    print(f"  {c['label']:16s} cos to condition mean: median {np.median(c['values']):.3f}")

# ── the stimulus sample the similarity matrix is drawn from ────────────────
PER = 100
rng_v = np.random.default_rng(1)
sel = np.concatenate([rng_v.choice(np.where(all_digits == d)[0], PER, replace=False)
                      for d in DIGIT_ORDER])

# ── condition means, digit-likeness, separability, PCA ─────────────────────
ROWS = ([dict(label=str(d), mean=F[all_digits == d].mean(0), color_key=f'digit:{d}')
         for d in DIGIT_ORDER] +
        [dict(label=str(d), mean=F_ood[ood_digits == d].mean(0), color_key='near_ood')
         for d in sorted(np.unique(ood_digits))] +
        [dict(label=FAR_LABEL[n], mean=F_far[n].mean(0), color_key='far_ood')
         for n in far_ood_data])
_n_ood_d = len(np.unique(ood_digits))
GROUP = [dict(label='trained', start=0, stop=4),
         dict(label='held-out', start=4, stop=4 + _n_ood_d),
         dict(label='far-OOD', start=4 + _n_ood_d, stop=len(ROWS))]

CENT = unit(np.stack([F[all_digits == d].mean(0) for d in DIGIT_ORDER]))
LIKE = [dict(label=lab, values=(unit(X) @ CENT.T).max(1), color_key=ck) for lab, X, ck in
        ([('ID test', F, 'id_data'), ('held-out digits', F_ood, 'near_ood')] +
         [(FAR_LABEL[n], F_far[n], 'far_ood') for n in far_ood_data])]

# digit separability: the fingerprint against activation controls. The network
# was never trained to tell 0 from 4 or 1 from 3 — its own output-side
# activations barely do, the fingerprint does.
REPS = [('BFT\nfingerprint', unit(F), F.shape[1], 'ours'),
        (r'$L_2$ act.',  layer_inputs[1], layer_inputs[1].shape[1], '0.55'),
        (r'$L_3$ act.',  layer_inputs[2], layer_inputs[2].shape[1], '0.55'),
        ('all act.',     np.concatenate(layer_inputs, axis=1),
         sum(li.shape[1] for li in layer_inputs), '0.55'),
        ('pixels',       layer_inputs[0], layer_inputs[0].shape[1], '0.55')]
SEP = [dict(label=lab, silhouette=silhouette_score(X, all_digits, metric='cosine'),
            knn=cross_val_score(KNeighborsClassifier(5, metric='cosine'), X,
                                all_digits, cv=5).mean(), dim=d, color_key=ck)
       for lab, X, d, ck in REPS]

_pca  = PCA(n_components=2).fit(unit(F))
rng_e = np.random.default_rng(3)


def _emb(X, n):
    idx = rng_e.choice(len(X), min(n, len(X)), replace=False)
    return _pca.transform(unit(X[idx]))


PCA_COORDS = dict(evr=_pca.explained_variance_ratio_[:2],
                  far=[dict(label=n, coords=_emb(F_far[n], 120)) for n in far_ood_data],
                  ood_coords=_emb(F_ood, 500),
                  digits=[dict(digit=d, coords=_emb(F[all_digits == d], 250))
                          for d in DIGIT_ORDER])

D = figdata.save('nb01_fingerprints', dict(
    digit_order=DIGIT_ORDER, dims=_dims, col_order=COL_ORDER, n_ev=N_EV,
    cols_even=COLS_EVEN, cols_odd=COLS_ODD,
    fp_mean_by_digit=np.stack([F[all_digits == d].mean(0) for d in DIGIT_ORDER]),
    fp_sel=F[sel], sel_per_digit=PER,
    sil_class=SIL_CLASS, sil_digit=SIL_DIGIT, r_ood=R_OOD,
    id_pts=np.array(id_pts), ood_pts=np.array(ood_pts), cond=COND,
    rows=ROWS, group=GROUP, like=LIKE, sep=SEP, rt_sims=rt_sims, pca=PCA_COORDS,
    # superset: the raw fingerprint matrices themselves (13-d, so cheap), which
    # is what any *new* fingerprint panel would need
    fp=dict(id=F.astype(np.float32), id_digits=all_digits, id_targets=all_targets,
            ood=F_ood.astype(np.float32), ood_digits=ood_digits, ood_preds=ood_preds,
            far=[dict(label=FAR_LABEL[n], F=F_far[n].astype(np.float32))
                 for n in far_ood_data])))
figdata.summary('nb01_fingerprints')


In [ ]:
# ── act baseline: the network's own activations on the SAME stimuli, row-aligned
#    with the fingerprint by construction (replaces add_activation_baselines.py).
from src import figdata
from src.separability import pool_activations as _pool
_reps = [{'label': '$L_2$ act.', 'X': _pool(layer_inputs[1]).astype(np.float32)},
         {'label': '$L_3$ act.', 'X': _pool(layer_inputs[2]).astype(np.float32)}]
for _r in _reps:
    _r['dim'] = int(_r['X'].shape[1])
_D = figdata.load('nb01_fingerprints')
_D['act'] = {'reps': _reps, 'aligned': 1,
             'index': np.arange(len(all_digits)), 'labels': np.asarray(all_digits)}
figdata.save('nb01_fingerprints', _D)
print('act baseline written:', [(r['label'], r['X'].shape) for r in _reps])

### 5b — Appendix figure

In [ ]:
# (appendix figure moved to notebooks/fig01_mlp_even_odd.ipynb)


## §6 — Hyperparameter check (held-out arbor R²)

Re-derives the per-layer circuit rank with the metric-free C0 rule (`src.hp_selection`), on the circuit tree's own arbors. Confirms the `K_MAX` in §1 sits at the reconstruction plateau; reads no fingerprint metric. Cached.

In [ ]:
from src import node_pos_arbor, nodes_per_layer, select_ranks, cached_result
_C = ANALYSIS_CTX
_npl = nodes_per_layer(_C['tree_circuit'], max_nodes=2)
_arbors = {li: [node_pos_arbor(nd, _C['layer_inputs'][li]) for nd in nds]
           for li, nds in _npl.items()}
hp_sel = cached_result(
    _C['tag'] + '_hpsel',
    lambda: select_ranks(_arbors, _C['labels_task'], k_cap=_C['k_cap'],
                         n_classes=_C['n_classes'], last_extra=_C['last_extra']),
    params=dict(kcap=_C['k_cap'], n=len(_C['labels_task']),
                kmax=list(_C['tree_circuit'].root.lambdas.shape)))
print('held-out K* per layer:', hp_sel['profile']['k_from_criterion'])
print('assembled profile     k_max=%s  n_branches=%s'
      % (hp_sel['profile']['k_max'], hp_sel['profile']['n_branches']))
print('§1 circuit k_max was :', _C.get('k_max_cfg'))

## §7 — Validation (faithfulness + class-relevant structure)

On the **circuit** tree: NMF init-stability per layer, causal-reconstruction fidelity (fc layers only), and the weight-term control (arbor-NMF vs activation-NMF separability). All via `src`; cached.

In [ ]:
# §7 — full validation suite -> logs/results/nb09_<exp>.json (figP_validation).
# Reuses src.validation; the circuit tree was traced with validate=True so causal
# reconstruction is available on fc layers. Cached; also writes the results JSON
# that scripts/build_validation_bundles.py re-encodes into the figure bundle.
import os, json as _json
from src import run_validation, cached_result
_C = ANALYSIS_CTX
val = cached_result(_C['tag'] + '_validation',
    lambda: run_validation(_C['exp'], _C['tree_circuit'], _C['tree_fp'],
                           _C['layer_inputs'], _C['labels_task'], _C['labels_fine'],
                           stab_seeds=_C.get('stab_seeds', 5)),
    params=dict(n=len(_C['labels_task']), exp=_C['exp'], fp='top2'))
_rd = os.path.join(REPO if 'REPO' in dir() else '..', 'logs', 'results')
os.makedirs(_rd, exist_ok=True)
with open(os.path.join(_rd, f"nb09_{_C['exp']}.json"), 'w') as _f:
    _json.dump(val, _f, indent=1)
print('validation written: logs/results/nb09_%s.json' % _C['exp'])
from src import validation_bundle
validation_bundle(_C['exp'], val, source=f"logs/results/nb09_{_C['exp']}.json")
print('figdata bundle written: nb09_%s_validation' % _C['exp'])
_st = val['stability']['per_layer']
print('  NMF stability/layer:', {k: round(v['mean'], 3) for k, v in _st.items()})
if val.get('recon'):
    print('  causal recon preact_R2 (median):', round(val['recon']['overall']['preact_r2']['median'], 3))
_bf = val['separability']['by_fine']
print('  separability by_fine: fp=%.3f  act(matched)=%.3f  (weight-term arbor=%.3f vs act=%.3f)'
      % (_bf['bft_fingerprint']['silhouette'], _bf['act_matched']['silhouette'],
         val['A1_weight_vs_activation']['fingerprint_separability']['arbor_nmf']['silhouette'],
         val['A1_weight_vs_activation']['fingerprint_separability']['activation_nmf']['silhouette']))

## §8 — Causal pruning

Prunes each class circuit's weights in BFT-importance order on the **circuit** tree and measures target vs bystander accuracy (`src.pruning`, wrapping `ablation_sweep`). One seed here; add checkpoints for the full seed×class grid on the cluster. Cached.

In [ ]:
# §8 — causal pruning -> data/results/<prune_name>.json (fig2e / fig6f / figB-d).
# Prunes each class circuit on the CIRCUIT tree; writes the per_obs JSON that
# scripts/build_pruning_bundle.py re-encodes. One seed here; the cluster run adds
# seeds via the loop below (extend _reps with more {seed, model, tree, targets}).
# All aggregation/tests live in src.bundles.pruning_bundle.
import os, json as _json
import numpy as _np
from src import run_pruning, pruning_results_dict, cached_result
_C = ANALYSIS_CTX
if not _C.get('prune_name'):
    print('pruning not wired for this model:', _C.get('skip_pruning') or _C['exp'])
    prune = None
else:
    _reps = [{'seed': 0, 'model': _C['model'], 'tree': _C['tree_circuit'],
              'layer_names': _C.get('layer_names'), 'targets': _C['labels_task']}]
    _targets = list(range(_C['n_classes']))
    _frac = _C.get('prune_fractions', (0.005, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5))
    _frac_stat = 0.2
    _ploader = _C.get('prune_eval_loader', _C['eval_loader'])
    prune = cached_result(_C['tag'] + '_pruning',
        lambda: run_pruning(_reps, _ploader, _targets,
                            fractions=_frac,
                            label_transform=_C.get('prune_label_transform', _C.get('label_transform')),
                            pred_transform=_C.get('pred_transform'),
                            device=_C.get('device'), n_random_repeats=_C.get('n_random', 5),
                            frac_stat=_frac_stat, verbose=1),
        params=dict(n=len(_targets), exp=_C['exp'], fractions=list(_frac),
                    seeds=[r['seed'] for r in _reps],
                    n_eval=len(getattr(_ploader, 'dataset', []) or []) or None))
    _rd = os.path.join(REPO if 'REPO' in dir() else '..', 'data', 'results')
    os.makedirs(_rd, exist_ok=True)
    _res = pruning_results_dict(_C['exp'], prune, fractions=_frac, frac_stat=_frac_stat)
    with open(os.path.join(_rd, _C['prune_name'] + '.json'), 'w') as _f:
        _json.dump(_res, _f, indent=1)
    print('pruning written: data/results/%s.json' % _C['prune_name'])
    from src import pruning_bundle
    pruning_bundle(_C['exp'], _res)   # -> figures/figdata (floors reject smoke runs)
    for m in ('bft_top', 'bft_bottom', 'random'):
        _d = [o['baseline'][str(o['target_class'])]
              - o['curves'][m][str(_frac_stat)][str(o['target_class'])]
              for o in prune['per_obs'] if m in o['curves']]
        if _d:
            print('  %-11s target drop@%.1f (mean): %+.3f' % (m, _frac_stat, _np.mean(_d)))

## §9 — Fingerprint separability (C1.8)

On the **fingerprint** tree: is the factor fingerprint more class-separable than the network's own activations, and where in the tree does that live? `src.separability` gives silhouette + kNN for the whole tree, its upper/lower slices, and the penultimate / full-activation baselines (native and dim-matched). Cached.

In [ ]:
from src import separability_evaluate, cached_result
_C = ANALYSIS_CTX
sep = cached_result(
    _C['tag'] + '_separability',
    lambda: separability_evaluate(_C['tree_fp'], _C['labels_fine'], _C['layer_inputs']),
    params=dict(n=len(_C['labels_fine']), tag='fp', fp='top2'))
_n = sep['native']
print('native silhouette:  fp_full=%.3f  output_only=%.3f  top_half=%.3f  spine=%.3f'
      % (_n['fp_full']['sil'], _n.get('fp_output_only', {}).get('sil', float('nan')),
         _n.get('fp_top_half', {}).get('sil', float('nan')), _n.get('fp_spine', {}).get('sil', float('nan'))))
print('activation baselines: penult=%.3f  full=%.3f'
      % (_n['act_penult']['sil'], _n['act_full']['sil']))
_p = sep['paired'].get('fp_full__vs__act_penult')
if _p:
    print('dim-matched @%d: fp(pca)=%.3f vs penult(pca)=%.3f'
          % (_p['match_dim'], _p['A_pca']['sil'], _p['B_pca']['sil']))